### Tools

Models can invoke tools to perform fetching of data, searching the web or running code.
1. A schema. including name,description and/or argument definations(mostly JSON schema)
2. A function or coroutine to execute

In [6]:
### Groq Model Integration
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]= os.getenv("GROQ_API_KEY")
model = init_chat_model(
    "openai/gpt-oss-120b",
    model_provider="groq",
)

response = model.invoke("Why do parrots talk?")
response.content


'Parrots aren’t “talking” in the same way humans do—they’re incredibly skilled mimics. Their ability to reproduce human speech is the result of a combination of anatomy, brain structure, social instincts, and learning opportunities. Here’s a breakdown of why some parrots can learn to “talk”:\n\n---\n\n## 1. The anatomy that makes mimicry possible  \n\n| Feature | What it does | Why it matters for speech |\n|---------|--------------|---------------------------|\n| **Syrinx** (the bird’s vocal organ) | Located at the base of the trachea, the syrinx has two sets of vibrating membranes that can be controlled independently. | Allows parrots to produce a wide range of frequencies, tones, and rapid modulations—far more than most other birds. |\n| **Highly mobile tongue and beak** | Parrots can shape sounds by moving their tongue, beak, and even the walls of their mouth. | Fine‑tunes the acoustic output, enabling them to approximate human phonemes. |\n| **Large, well‑developed brain region (th

In [20]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the weather for a given location."""
    return f"The current weather in {location} is sunny with a temperature of 25°C."

model_with_tools=model.bind_tools([get_weather])

In [21]:
response=model_with_tools.invoke("What is the weather in New York?")
print(response.content)
for tool_call in response.tool_calls:
    print(f"Tool:{tool_call['name']}")
    print(f"Args:{tool_call['args']}")


Tool:get_weather
Args:{'location': 'New York'}


### Tool Execution Loops

In [22]:
# Step 1: Model generates tool calls
messages= [{"role":"user", "content":"Whats the weather is Boston?"}]
ai_msg=model_with_tools.invoke(messages)
messages.append(ai_msg)

#Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result= get_weather.invoke(tool_call)
    messages.append(tool_result)
    
# Step 3: Pass results back to model for final response
final_response= model_with_tools.invoke(messages)
print(final_response.text)


The current weather in Boston is sunny with a temperature of 25 °C.


In [23]:
messages

[{'role': 'user', 'content': 'Whats the weather is Boston?'},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks: "Whats the weather is Boston?" Likely they want current weather in Boston. Use function get_weather.', 'tool_calls': [{'id': 'fc_78faeb5e-f9e9-4ef8-aec2-f1e2b97489e3', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 127, 'total_tokens': 179, 'completion_time': 0.109206215, 'completion_tokens_details': {'reasoning_tokens': 25}, 'prompt_time': 0.047050229, 'prompt_tokens_details': None, 'queue_time': 0.35938842, 'total_time': 0.156256444}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_6c36ac20de', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0b2fc-2ba3-7012-bf6a-8a449a053877-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Bo